# 🧪 atomipy Visual Builder - Google Colab GPU Launch Guide
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mholmboe/atomipy-web-module/blob/main/ColabLaunchGuide.ipynb)

Welcome to the official launch manual for running the **atomipy Visual Builder** on **Google Colab** with full **GPU hardware acceleration**!

This notebook launches the visual interface in Colab's cloud environment so you can build complex mineral-water systems and run **OpenMM molecular dynamics simulations** on a high-performance **NVIDIA T4, L4, or A100 GPU**.

Unlike the public site at [www.atomipy.io](https://www.atomipy.io) (which runs on CPU only), GPU simulations are **enabled** here.

---

## ⚡ STEP 0: Enable GPU Acceleration

1. In the top-right menu of Colab, click **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (or a higher tier if available) under *Hardware accelerator*.
3. Click **Save**.

## 📦 STEP 1: Clone and Build the Application

This wipes any previous install, clones the repo, installs the Python dependencies (FastAPI + OpenMM + atomipy deps), and builds the React/Vite production frontend bundle.

In [ ]:
# 1. Reset directory and wipe previous folders
%cd /content
!rm -rf atomipy-web-module

# 2. Clone fresh
!git clone https://github.com/mholmboe/atomipy-web-module.git
%cd atomipy-web-module

# 3. Install Python dependencies (FastAPI backend + OpenMM + atomipy deps)
!pip install -q -r requirements.txt

# 4. Install Node and build the React frontend (served by FastAPI)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install --legacy-peer-deps
!npm run build

# 5. Localtunnel for the public URL
!npm install -g localtunnel

## 🧪 STEP 1b (OPTIONAL): Enable the Organic Molecule (GAFF/OpenFF) node

The **Organic Molecule** node parametrizes small molecules with GAFF / OpenFF
via a separate **OpenFF worker**. Run the cell below **only if you need that
node** — it installs the OpenFF + ACPYPE stack with **micromamba** (no kernel
restart) and starts the worker on port `8001`. It adds a few minutes the first
time. **Skip it** if you only build systems, assign MINFF/CLAYFF, and run MD/EM.

> Run this **before** Step 2. All other nodes work without it.

In [ ]:
# (Optional) OpenFF/ACPYPE worker for the Organic Molecule node.
import os, subprocess, time, json, urllib.request

REPO = "/content/atomipy-web-module"
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"
MAMBA = "/content/bin/micromamba"

# 1. Standalone micromamba binary (no conda install, no kernel restart)
if not os.path.exists(MAMBA):
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xj -C /content bin/micromamba",
        shell=True, check=True,
    )

# 2. Create the atomipy-openff environment (openff-toolkit, acpype, rdkit, openbabel, ...)
env_python = "/content/micromamba/envs/atomipy-openff/bin/python"
if not os.path.exists(env_python):
    print("Installing OpenFF/ACPYPE stack - this takes a few minutes...")
    subprocess.run(
        [MAMBA, "create", "-y", "-f", f"{REPO}/envs/atomipy-openff.yml"],
        check=True,
    )

# 3. Launch the OpenFF worker on 127.0.0.1:8001 in the background
wenv = os.environ.copy()
wenv["PYTHONPATH"] = f"{REPO}/workers/openff_worker"
wenv["INTERCHANGE_EXPERIMENTAL"] = "1"
subprocess.Popen(
    [MAMBA, "run", "-n", "atomipy-openff",
     "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8001"],
    cwd=f"{REPO}/workers/openff_worker", env=wenv,
    stdout=open("/content/openff_worker.log", "w"), stderr=subprocess.STDOUT,
)

# 4. Wait for the worker to be ready (the main server uses OPENFF_WORKER_URL,
#    which defaults to http://127.0.0.1:8001 - no extra config needed).
for _ in range(90):
    try:
        s = urllib.request.urlopen("http://127.0.0.1:8001/status", timeout=2).read()
        print("\u2705 OpenFF worker ready (acpype_available =",
              json.loads(s).get("acpype_available"), ")")
        break
    except Exception:
        time.sleep(3)
else:
    print("\u26a0\ufe0f Worker not up yet - check /content/openff_worker.log")

## 🚀 STEP 2: Launch the Tunnel and Visual Builder

This cell:
1. Retrieves your Colab instance's **external IP address** (your **Tunnel Password**).
2. Opens a secure Localtunnel on port `5002`.
3. Boots the **FastAPI** server, which serves both the API *and* the built frontend, with simulations **enabled** (GPU).

### Instructions
- 🔑 **Copy the IP address** printed below.
- 🔗 **Click the generated `Localtunnel` link**.
- 🔓 Paste the IP into the **"Tunnel Password"** field in your browser and submit.

> The **Organic Molecule (GAFF/OpenFF)** node works here only if you ran the optional **Step 1b** above; otherwise that single node is unavailable (everything else still works).

In [ ]:
import os
import subprocess
import time
import urllib.request

REPO = "/content/atomipy-web-module"
PORT = "5002"

# 1. Get this instance's IP (used as the Localtunnel password)
colab_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()

# 2. Start localtunnel in the background
lt_proc = subprocess.Popen(["lt", "--port", PORT], stdout=subprocess.PIPE, text=True)
time.sleep(3)
public_url = "Generating URL..."
for line in lt_proc.stdout:
    if "your url is:" in line:
        public_url = line.split("your url is:")[-1].strip()
        break

print("\n==================================================")
print(f"🔑 STEP 1 - COPY THIS PASSWORD: {colab_ip}")
print(f"👉 STEP 2 - CLICK THIS LINK: {public_url}")
print("==================================================\n")

# 3. Launch the FastAPI server (serves frontend + API; simulations enabled).
#    DISABLE_SIMULATION is left unset so MD/EM runs on the Colab GPU.
#    OPENFF_WORKER_URL defaults to http://127.0.0.1:8001 (Step 1b worker).
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO}:{REPO}/backend/core"
env["FRONTEND_DIST"] = f"{REPO}/dist"
subprocess.run(
    [
        "uvicorn", "main:app",
        "--app-dir", f"{REPO}/backend/core",
        "--host", "0.0.0.0",
        "--port", PORT,
    ],
    env=env,
)